# Price Watch: test scrapers and explore the data

Run the cells top to bottom (**Shift+Enter**). Sections 1–3 only read the competitors' websites.
Sections 4–5 use the database, so they need your Supabase connection string in a `.env` file in the project
folder (see `SUPABASE_SETUP.md`) and `pip install psycopg2-binary` once in the Anaconda Prompt.

In [ ]:
# Setup: point Python at the project folder (the folder above this notebook)
import os, sys, importlib
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
from scrapers import foodplus, naivas, quickmart
for m in (foodplus, naivas, quickmart):
    importlib.reload(m)     # picks up edits to the scraper files without restarting the kernel
pd.set_option("display.max_colwidth", 70)
print("Project folder:", ROOT)

## 1. Each competitor's category tree
This is what a full-catalogue crawl (`page_type = all` in `config/targets.csv`) goes through.
Every product is stored with its site category path.

In [ ]:
cats = pd.DataFrame(foodplus.discover_categories(), columns=["site_category", "url"])
print(len(cats), "Foodplus categories")
cats[cats.site_category.str.contains("Electronics")].head(20)

In [ ]:
cats_nv = pd.DataFrame(naivas.discover_categories(), columns=["site_category", "url"])
print(len(cats_nv), "Naivas categories")
cats_nv.head(20)

## 2. One category, as a table
Any category URL from above (all its pages are followed).

In [ ]:
url = "https://foodplus.co.ke/electronics/small-appliances/kettles.html"

df = pd.DataFrame(foodplus.scrape_category(url, None, "Electronics > Small Appliances > Kettles"))
df["discount_%"] = ((df.normal_price - df.promo_price) / df.normal_price * 100).round(1)
print(f"{len(df)} products, {df.promo_price.notna().sum()} on promo")
df[["sku", "name", "normal_price", "promo_price", "discount_%", "in_stock"]].head(15)

## 3. One product page

In [ ]:
next(foodplus.scrape_product("https://foodplus.co.ke/ramtons-rm-226-corded-kettle-1-7l.html", None))

## 4. Run a crawl into the database (optional)
The daily GitHub run does this for you. Use it to try a change: `--max-categories 3` crawls just the first
3 categories, which takes about a minute. Remove it for the whole catalogue (25–60 minutes per site).

In [ ]:
RUN = False   # change to True to run it

if RUN:
    from pricewatch import run
    run.main(["--only", "foodplus", "--max-categories", "3"])
else:
    print("Skipped. Set RUN = True to run it.")

## 5. Look at the data in the database
The same views Power BI reads.

In [ ]:
con = None
try:
    from pricewatch import db
    con = db.connect()
except ModuleNotFoundError:
    print("psycopg2 isn't installed yet. In the Anaconda Prompt run:  pip install psycopg2-binary  then restart the kernel.")
except SystemExit as e:
    print(e)

if con:
    q = lambda sql: pd.read_sql(sql, con)
    display(q("SELECT competitor, COUNT(*) AS products, MAX(last_seen) AS last_seen FROM v_products GROUP BY 1"))
    display(q("""SELECT competitor, site_category_1, COUNT(*) AS products
                 FROM v_products GROUP BY 1, 2 ORDER BY 1, 3 DESC""").head(30))

In [ ]:
if con:
    display(q("""SELECT p.competitor, p.name, c.date, c.change_label, c.from_price, c.to_price, c.change_pct
                 FROM v_price_changes c JOIN v_products p USING (product_id)
                 ORDER BY c.date DESC LIMIT 30"""))

In [ ]:
if con:
    # biggest promos right now, any competitor, any category
    display(q("""SELECT p.competitor, p.site_category, p.name, d.normal_price, d.promo_price, d.promo_discount_pct
                 FROM v_daily_prices d JOIN v_products p USING (product_id)
                 WHERE d.date = (SELECT MAX(date) FROM v_daily_prices) AND d.on_promo = 1
                 ORDER BY d.promo_discount_pct DESC LIMIT 25"""))

## 6. Starting a scraper for another site
Paste a product URL from another competitor to see how it shows prices.

In [ ]:
from bs4 import BeautifulSoup
from scrapers._utils import get

test_url = "https://foodplus.co.ke/ramtons-rm-226-corded-kettle-1-7l.html"
soup = BeautifulSoup(get(test_url).text, "html.parser")
print("Title:", soup.title.get_text(strip=True) if soup.title else None)
for sel in ['[data-price-type]', '[itemprop="price"]', '[itemprop="sku"]', '.price', '.old-price', '.special-price',
            '[class*="line-through"]', 'script[type="application/ld+json"]']:
    found = soup.select(sel)
    print(f"\n{sel}: {len(found)} found")
    for el in found[:3]:
        print("   ", str(el)[:200].replace("\n", " "))